<a href="https://colab.research.google.com/github/mahendrapd/RAG-Model/blob/main/AryuProg1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries required to install before executing this model
1. pip install numpy
2. pip install pandas
3. pip install torch
4. pip install PyPDF
5. pip install faiss-gpu
6. pip install tranformers
7. pip install sentence-transformers
8. pip install sentencepiece


In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [4]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/AryuvedaR/my_packages")
# package_path = "/content/drive/MyDrive/AryuvedaR/my_packages"

# if package_path not in sys.path:
#     sys.path.insert(0, package_path)


In [34]:
from huggingface_hub import login

In [35]:
login()

In [ ]:
pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.6 MB/s eta 0:00:00


In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [106]:
from pypdf import PdfReader

def read_pdf_pypdf(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""  # Handle pages with no text
        return text.strip()
    except FileNotFoundError:
        return "Error: File not found."
    except Exception as e:
        return f"Error reading PDF: {e}"

# Example usage
if __name__ == "__main__":
    pdf_text = read_pdf_pypdf("/content/drive/MyDrive/AryuvedaR/SushrutaS.pdf")
    #print(pdf_text)

In [107]:
def naive_code_chunk(text, chunk_size=300):
    """Slices plain text without considering code structure."""
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

simple_chunks = naive_code_chunk(pdf_text, chunk_size=300)

In [108]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [109]:
embeddings = model.encode(
    simple_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,  # Useful for cosine similarity
    show_progress_bar=True
)

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

In [110]:
print("Embedding shape:", embeddings.shape)
print("First embedding length:", len(embeddings[0]))

Embedding shape: (2604, 384)
First embedding length: 384


In [111]:
embeddings = embeddings.astype("float32")

In [112]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

In [113]:
index.add(embeddings)

In [114]:
faiss.write_index(index, "/content/drive/MyDrive/AryuvedaR/SushrutaS_index.faiss")

In [ ]:
#index = faiss.read_index("chunk_index.faiss")

In [115]:
print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 2604


In [116]:
query = input("Enter your query: ")

# Embed the query
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Search top 3 similar chunks
k = 5
scores, indices = index.search(query_embedding, k)

Enter your query: What is Ojas?


In [117]:
retrieved_chunks = [simple_chunks[i] for i in indices[0]]

In [118]:
print("Retrieved Context:\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}\n")

Retrieved Context:

1. ntityofsemenand(ojas)albumenintheorganism.Tuemeatcookedandpreparedintheforegoingmanner,whenmincedandmadeintocakes,iscalledUllupta-Mansam,whichresemblestheabove-saidParishushka-Mansaminitsproperties,thoughitislighterthanthelatterowingtothefactofitshavingbeencookedoveracharcoalfire.Thesamemeat,cookedo

2. allactsofvitality,improvesthevoiceandcomplexion,andhelpsboththeexternal(operative")andtheinternal^intellectual)senseorgans,indulyperformingtheirnaturalfunctions.Authoritativeversesonthesub-ject:—Ojas(albumenbeingofawhitecolourbelongstotheclassofSomatmakam(cooling)substances.
*TheSanskritlerm"Ojas"'h

3. bumen(ojas).Itisknowntobringaboutanadhesionandreductionoffracturedanddislocatedbones,andincreasesthequantityofsemenandoja(albumen)insubjectsfoundwantinginthosetwoimportantprinciplesofhfe.Meat-soup,preparedwiththejuiceoftheDadima,etc.,andseasonedwithpungentcondiments,increasesthequantityofsemenandten

4. edalbumen(Ojas)ischaracterisedfirstlybyitsdislodgmentfromitspr

In [119]:
context = "\n\n".join(retrieved_chunks)

In [120]:
augmented_prompt = f"""
You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

In [121]:
augmented_prompt

'\nYou are a helpful assistant.\n\nUse the following context to answer the user\'s question.\nIf the answer is not contained in the context, say you don\'t know.\n\nContext:\nntityofsemenand(ojas)albumenintheorganism.Tuemeatcookedandpreparedintheforegoingmanner,whenmincedandmadeintocakes,iscalledUllupta-Mansam,whichresemblestheabove-saidParishushka-Mansaminitsproperties,thoughitislighterthanthelatterowingtothefactofitshavingbeencookedoveracharcoalfire.Thesamemeat,cookedo\n\nallactsofvitality,improvesthevoiceandcomplexion,andhelpsboththeexternal(operative")andtheinternal^intellectual)senseorgans,indulyperformingtheirnaturalfunctions.Authoritativeversesonthesub-ject:—Ojas(albumenbeingofawhitecolourbelongstotheclassofSomatmakam(cooling)substances.\n*TheSanskritlerm"Ojas"\'h\n\nbumen(ojas).Itisknowntobringaboutanadhesionandreductionoffracturedanddislocatedbones,andincreasesthequantityofsemenandoja(albumen)insubjectsfoundwantinginthosetwoimportantprinciplesofhfe.Meat-soup,preparedwiththejui

In [122]:
from transformers import pipeline

In [123]:
import re

In [124]:
generator = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it"
)

response = generator(
    augmented_prompt,
    max_new_tokens=800,
    do_sample=False,
    eos_token_id=generator.tokenizer.eos_token_id,
)
text = response[0]["generated_text"]

sentences = re.findall(r'.*?[.!?](?=\s|$)', text, flags=re.DOTALL)
text = "".join(sentences)

print(text)
#print(response[0]["generated_text"])

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
ntityofsemenand(ojas)albumenintheorganism.Tuemeatcookedandpreparedintheforegoingmanner,whenmincedandmadeintocakes,iscalledUllupta-Mansam,whichresemblestheabove-saidParishushka-Mansaminitsproperties,thoughitislighterthanthelatterowingtothefactofitshavingbeencookedoveracharcoalfire.Thesamemeat,cookedo

allactsofvitality,improvesthevoiceandcomplexion,andhelpsboththeexternal(operative")andtheinternal^intellectual)senseorgans,indulyperformingtheirnaturalfunctions.Authoritativeversesonthesub-ject:—Ojas(albumenbeingofawhitecolourbelongstotheclassofSomatmakam(cooling)substances.
*TheSanskritlerm"Ojas"'h

bumen(ojas).Itisknowntobringaboutanadhesionandreductionoffracturedanddislocatedbones,andincreasesthequantityofsemenandoja(albumen)insubjectsfoundwantinginthosetwoimportantprinciplesofhfe.Meat-soup,preparedwiththejuiceoftheDadima,et